# 04a — Activation Functions

Corresponds to: Note 04 — Activation Functions

**Libraries introduced in this notebook:** No new libraries — numpy only (introduced in Note 03).

**What this notebook demonstrates:**
1. Why the step function breaks backpropagation — chain rule with zero derivative
2. Sigmoid — formula, derivative derivation, vanishing gradient across layers
3. Tanh — formula, derivative, comparison with sigmoid
4. ReLU — formula, derivative, sparse activation
5. Leaky ReLU — formula, derivative, comparison with ReLU
6. Softmax — formula, probability distribution, temperature effect

---
## Part 1: Why the Step Function Breaks Backpropagation

The Heaviside step function has derivative zero everywhere. We show concretely what this means for the chain rule in a 2-layer network.

The chain rule for a hidden weight w⁽¹⁾ is:

```
∂L/∂w⁽¹⁾ = dL/dŷ · f'(z⁽²⁾) · w⁽²⁾ · f'(z⁽¹⁾) · x
```

If f is the step function, f'(z) = 0 everywhere — the entire product is zero.

In [ ]:
import numpy as np

def step(z):
    return (z >= 0).astype(float)

def step_deriv(z):
    return np.zeros_like(z)  # 0 everywhere

# Simulate one chain rule product through a hidden layer
dL_dyhat = 1.0      # gradient at output
f_prime_z2 = step_deriv(np.array([0.5]))[0]   # derivative at output layer
w2 = 1.0            # weight in layer above
f_prime_z1 = step_deriv(np.array([0.3]))[0]   # derivative at hidden layer
x1 = 1.0            # input value

gradient = dL_dyhat * f_prime_z2 * w2 * f_prime_z1 * x1

print('Chain rule factors:')
print(f'  dL/dŷ        = {dL_dyhat}')
print(f"  f'(z⁽²⁾)     = {f_prime_z2}  ← step derivative")
print(f'  w⁽²⁾         = {w2}')
print(f"  f'(z⁽¹⁾)     = {f_prime_z1}  ← step derivative")
print(f'  x₁           = {x1}')
print()
print(f'∂L/∂w⁽¹⁾ = {gradient}')
print()
print('Zero gradient — hidden weight receives no update, regardless of the error.')

---
## Part 2: Sigmoid

Formula: σ(z) = 1 / (1 + e⁻ᶻ)

Derivative: dσ/dz = σ(z) · (1 − σ(z))

We implement both, verify the worked example from the note, then show the vanishing gradient across multiple layers.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Worked example from note: z = 2.0
z = 2.0
s = sigmoid(z)
ds = sigmoid_deriv(z)

print('Sigmoid — worked example (z = 2.0):')
print(f'  σ(2.0)   = 1 / (1 + e⁻²) = {s:.4f}')
print(f'  dσ/dz    = {s:.4f} × (1 − {s:.4f}) = {ds:.4f}')
print()

# Maximum derivative
z0 = 0.0
print(f'Maximum derivative at z = 0: σ(0) × (1 − σ(0)) = {sigmoid_deriv(z0):.4f}')
print()

# Vanishing gradient across L layers — all at maximum derivative 0.25
print('Vanishing gradient — gradient reaching layer 1 = 0.25^L:')
for L in [1, 2, 4, 8, 10]:
    grad = 0.25 ** L
    print(f'  L = {L:2d} layers:   0.25^{L} = {grad:.10f}')

---
## Part 3: Tanh

Formula: tanh(z) = (eᶻ − e⁻ᶻ) / (eᶻ + e⁻ᶻ)

Derivative: d(tanh)/dz = 1 − tanh²(z)

We verify the worked example from the note and compare the derivative range with sigmoid.

In [ ]:
def tanh_deriv(z):
    return 1 - np.tanh(z) ** 2

# Worked example from note: z = 2.0
z = 2.0
t = np.tanh(z)
dt = tanh_deriv(z)

print('Tanh — worked example (z = 2.0):')
print(f'  tanh(2.0)      = {t:.4f}')
print(f'  d(tanh)/dz     = 1 − {t:.4f}² = 1 − {t**2:.4f} = {dt:.4f}')
print()

# Maximum derivative at z = 0
print(f'Maximum derivative at z = 0: 1 − tanh(0)² = {tanh_deriv(0.0):.4f}')
print()

# Saturation comparison
print('Saturation — derivative at large |z|:')
print(f'  {'z':>5}   {'sigmoid deriv':>15}   {'tanh deriv':>12}')
print(f'  {"-"*5}   {"-"*15}   {"-"*12}')
for z_val in [0.0, 1.0, 2.0, 3.0, 5.0]:
    print(f'  {z_val:>5.1f}   {sigmoid_deriv(z_val):>15.6f}   {tanh_deriv(z_val):>12.6f}')

---
## Part 4: ReLU

Formula: f(z) = max(0, z)

Derivative: f'(z) = 1 if z > 0, else 0

We verify the worked example from the note and demonstrate sparse activation on a hidden layer.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

# Worked example from note
test_z = np.array([3.5, 0.0, -2.1])

print('ReLU — worked example:')
print(f'  {"z":>6}   {"f(z)":>8}   {"f\'(z)":>8}')
print(f'  {"-"*6}   {"-"*8}   {"-"*8}')
for z_val in test_z:
    print(f'  {z_val:>6.1f}   {relu(z_val):>8.4f}   {relu_deriv(z_val):>8.4f}')
print()

# Gradient preservation vs sigmoid
print('Gradient across 10 layers — ReLU (z > 0) vs sigmoid (max derivative):')
grad_relu    = 1.0
grad_sigmoid = 1.0
print(f'  {"Layer":>7}   {"ReLU gradient":>15}   {"Sigmoid gradient":>18}')
print(f'  {"-"*7}   {"-"*15}   {"-"*18}')
for l in range(1, 11):
    grad_relu    *= 1.0
    grad_sigmoid *= 0.25
    print(f'  {l:>7}   {grad_relu:>15.6f}   {grad_sigmoid:>18.10f}')

---
## Part 5: Leaky ReLU

Formula: f(z) = z if z > 0, else αz  (α = 0.01)

Derivative: f'(z) = 1 if z > 0, else α

We verify the worked example and show that the gradient is never exactly zero — fixing the dying neuron problem.

In [ ]:
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

# Worked example from note
test_z = np.array([3.5, 0.0, -2.1])
alpha  = 0.01

print(f'Leaky ReLU — worked example (α = {alpha}):')
print(f'  {"z":>6}   {"ReLU f(z)":>10}   {"Leaky f(z)":>12}   {"ReLU f\'(z)":>12}   {"Leaky f\'(z)":>13}')
print(f'  {"-"*6}   {"-"*10}   {"-"*12}   {"-"*12}   {"-"*13}')
for z_val in test_z:
    print(f'  {z_val:>6.1f}   {relu(z_val):>10.4f}   {leaky_relu(z_val):>12.4f}   '
          f'{relu_deriv(z_val):>12.4f}   {leaky_relu_deriv(z_val):>13.4f}')
print()

# Dying neuron scenario
print('Dying neuron scenario — neuron stuck at z = −3.0:')
z_dead = -3.0
upstream = 0.5
grad_relu   = relu_deriv(z_dead)   * upstream
grad_leaky  = leaky_relu_deriv(z_dead) * upstream
print(f'  Upstream gradient:          {upstream}')
print(f'  ReLU gradient through:      {relu_deriv(z_dead)} × {upstream} = {grad_relu}  ← dead')
print(f'  Leaky ReLU gradient through: {leaky_relu_deriv(z_dead)} × {upstream} = {grad_leaky}  ← can recover')

---
## Part 6: Softmax

Formula: softmax(zₖ) = e^zₖ / Σⱼ e^zⱼ

We verify the worked example from the note and demonstrate the temperature effect.

In [ ]:
def softmax(z, T=1.0):
    e = np.exp(z / T)
    return e / e.sum()

# Worked example from note: z = [2.0, 1.0, 0.5]
z = np.array([2.0, 1.0, 0.5])
exp_z = np.exp(z)
probs = softmax(z)

print('Softmax — worked example (z = [2.0, 1.0, 0.5]):')
print(f'  e^z = {exp_z.round(3)}')
print(f'  sum = {exp_z.sum():.3f}')
print()
for k, (zk, ek, pk) in enumerate(zip(z, exp_z, probs), 1):
    print(f'  softmax(z{k}) = {ek:.3f} / {exp_z.sum():.3f} = {pk:.3f}')
print(f'  sum of probs = {probs.sum():.3f}  ✓')
print()

# Temperature effect
print('Temperature effect on softmax(z = [2.0, 1.0, 0.5]):')
print(f'  {"T":>6}   {"Class 1":>10}   {"Class 2":>10}   {"Class 3":>10}   {"Interpretation":>25}')
print(f'  {"-"*6}   {"-"*10}   {"-"*10}   {"-"*10}   {"-"*25}')
configs = [
    (0.1,  'near one-hot'),
    (1.0,  'standard'),
    (5.0,  'softer'),
    (100.0,'near uniform'),
]
for T, label in configs:
    p = softmax(z, T)
    print(f'  {T:>6.1f}   {p[0]:>10.4f}   {p[1]:>10.4f}   {p[2]:>10.4f}   {label:>25}')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| Step function + chain rule | f'(z) = 0 kills the gradient product — hidden weights never update |
| Sigmoid derivative | dσ/dz = σ(z)(1 − σ(z)), max 0.25 — gradient shrinks by ×0.25 per layer |
| Vanishing gradient | 0.25^10 ≈ 10⁻⁶ — gradient effectively zero after 10 sigmoid layers |
| Tanh derivative | 1 − tanh²(z), max 1.0 at z = 0 — better than sigmoid but still saturates |
| ReLU gradient | f'(z) = 1 for z > 0 — gradient preserved exactly, no shrinkage |
| Dying ReLU | f'(z) = 0 for z < 0 — neuron stuck at z < 0 receives zero gradient forever |
| Leaky ReLU fix | f'(z) = α for z ≤ 0 — small non-zero gradient keeps neuron recoverable |
| Softmax | Probabilities sum to 1.0 ✓ — temperature T controls sharpness of distribution |